In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.ecommerce_orders_cdf;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.ecommerce_orders_cdf(
    order_id INT,
    customer_name STRING,
    order_status STRING,
    amount INT
)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

Insert Intial Records

In [0]:
%sql
INSERT INTO workspace.default.ecommerce_orders_cdf VALUES
(101, 'Arun', 'Pending', 500),
(102, 'Priya', 'Completed', 800),
(103, 'John', 'Pending', 300);

#### Verify data

In [0]:
%sql
SELECT *
FROM workspace.default.ecommerce_orders_cdf
ORDER BY order_id;
     

#### Update Existing Record


In [0]:
%sql
UPDATE workspace.default.ecommerce_orders_cdf
SET order_status = 'Shipped'
WHERE order_id = 101;

#### Read Changes

In [0]:
%sql
DESCRIBE HISTORY workspace.default.ecommerce_orders_cdf;

In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
ORDER BY _commit_version, _change_type;

#### Insert another order and delete an existing order

Insert order 104

In [0]:
%sql
INSERT INTO workspace.default.ecommerce_orders_cdf VALUES
(104, 'David', 'Pending', 600);

Delete order 103

In [0]:
%sql
DELETE FROM workspace.default.ecommerce_orders_cdf
WHERE order_id = 103;

In [0]:
%sql
SELECT *
FROM workspace.default.ecommerce_orders_cdf
ORDER BY order_id;
     

In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
ORDER BY _commit_version, order_id, _change_type;

#### Version-based incremental processing

Suppose your pipeline has already processed all changes through Delta version 2

In [0]:
%sql
SELECT *
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    3
);

#### Update Arun's order using MERGE INTO

In [0]:
%sql
MERGE INTO workspace.default.ecommerce_orders_cdf AS target
USING (
    SELECT
        101 AS order_id,
        'Arun' AS customer_name,
        'Delivered' AS order_status,
        500 AS amount
) AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET
        target.order_status = source.order_status;

#### Check the changes captured by CDF

In [0]:
%sql
SELECT
    order_id,
    customer_name,
    order_status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.default.ecommerce_orders_cdf',
    1
)
WHERE order_id = 101
ORDER BY _commit_version, _change_type;